<a href="https://colab.research.google.com/github/yantavares/machado-gpt/blob/main/notebooks/machado_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT de Machado de Assis com o nanoGPT

## Redes Neurais e Aprendizado Profundo: Projeto Individual 1.

Yan Tavares de Oliveira, 252104296.

| Marca | Autoria |
|---|---|
| `# [AUTORIA: TERCEIROS]` | nanoGPT, Andrej Karpathy (MIT) |
| `# [AUTORIA: ALUNO]` | código do aluno |
| `# >>> MODIFICAÇÃO DO ALUNO` | alteração do aluno dentro do nanoGPT |
| `# [ALUNO]` | alteração do aluno dentro do `gpt.py` do vídeo de referência |

Ambiente: Colab com GPU T4.

## 1. Ambiente

In [1]:
# [AUTORIA: ALUNO]
import os, json, torch
print(torch.__version__, "CUDA:", torch.cuda.is_available())
!pip -q install tiktoken transformers
os.makedirs("/content/projeto", exist_ok=True)
%cd /content/projeto
!mkdir -p machado_gpt experiments/configs corpus results tests

2.11.0+cu128 CUDA: True
/content/projeto


## 2. nanoGPT

In [2]:
# [AUTORIA: TERCEIROS] nanoGPT, Andrej Karpathy (MIT)
!rm -rf nanoGPT-master && git clone -q https://github.com/karpathy/nanoGPT nanoGPT-master
!cd nanoGPT-master && git checkout -q 3adf61e154c3fe3fca428ad6bc3818b27a3b8291 && git log -1 --oneline

3adf61e (HEAD, origin/master, origin/HEAD, master) Update README to mention nanochat and deprecation


In [3]:
# [AUTORIA: ALUNO] única alteração no nanoGPT: carregar checkpoints GPT-2 do Hugging Face (GPorTuguese-2)
from pathlib import Path

def troca(arquivo, de, ate, novo):
    p = Path("nanoGPT-master") / arquivo
    s = p.read_text()
    i = s.index(de)
    p.write_text(s[:i] + novo + s[s.index(ate, i):])

troca('model.py', '        assert model_type in {', '        override_args = override_args or {}', '''\
        # >>> MODIFICAÇÃO DO ALUNO (projeto Machado de Assis) -----------------
        oficiais = {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        assert model_type in oficiais or '/' in model_type
        # <<< FIM DA MODIFICAÇÃO -----------------------------------------------
''')
troca('model.py', '        # n_layer, n_head and n_embd are determined', '        # we can override the dropout rate', '''\
        # >>> MODIFICAÇÃO DO ALUNO: ramo para checkpoints de terceiros ---------
        if model_type not in oficiais:
            from transformers import GPT2Config
            hf = GPT2Config.from_pretrained(model_type)
            config_args = dict(n_layer=hf.n_layer, n_head=hf.n_head, n_embd=hf.n_embd,
                               vocab_size=hf.vocab_size, block_size=hf.n_positions, bias=True)
            print(f"config from checkpoint: {config_args}")
        else:
        # <<< FIM DA MODIFICAÇÃO (o bloco original abaixo só ganhou um nível de indentação)
            # n_layer, n_head and n_embd are determined from model_type
            config_args = {
                'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
                'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
                'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
                'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
            }[model_type]
            print("forcing vocab_size=50257, block_size=1024, bias=True")
            config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
            config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
            config_args['bias'] = True # always True for GPT model checkpoints
''')
troca('train.py', "elif init_from.startswith('gpt2'):", '    print(f"Initializing from OpenAI GPT-2 weights', '''\
# >>> MODIFICAÇÃO DO ALUNO: também aceita ids do Hugging Face com arquitetura GPT-2
elif init_from.startswith('gpt2') or '/' in init_from:
# <<< FIM DA MODIFICAÇÃO
''')
!cd nanoGPT-master && git diff --stat

 model.py | 36 ++++++++++++++++++++++++------------
 train.py |  4 +++-
 2 files changed, 27 insertions(+), 13 deletions(-)


## 3. Corpus

In [4]:
# [AUTORIA: ALUNO]
!wget -q -O machado.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/machado.zip
!rm -rf machado && unzip -q machado.zip && find machado -name "*.txt" | wc -l

246


In [5]:
%%writefile machado_gpt/__init__.py
# [AUTORIA: ALUNO]
"""Código autoral do projeto "GPT de Machado de Assis" (RNAP 2026/2, UnB)."""

Writing machado_gpt/__init__.py


In [6]:
%%writefile machado_gpt/corpus.py
# [AUTORIA: ALUNO]
"""Constrói o corpus de treino a partir da Obra Completa de Machado de Assis.

Autoria: código original do aluno.
"""

import argparse
import csv
import hashlib
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

RAIZ = Path(__file__).resolve().parent.parent
ACERVO = RAIZ / "machado"
SAIDA = RAIZ / "corpus"

GENEROS = {
    "romance": "romance",
    "contos": "conto",
    "cronica": "crônica",
    "critica": "crítica",
    "poesia": "poesia",
    "teatro": "teatro",
    "miscelanea": "miscelânea",
    "traducao": "tradução",
}

CABECALHO_MANUAL = {
    "critica/mact23.txt": 6,
    "critica/mact28.txt": 6,
    "critica/mact31.txt": 6,
    "critica/mact37.txt": 6,
    "critica/mact45.txt": 6,
    "poesia/maps07.txt": 7,
}

TAM_BLOCO = 4000
CICLO = 20
MIN_DUPLICATA = 100


def normaliza(texto):
    """Normaliza espaços e a forma Unicode de um parágrafo."""
    texto = unicodedata.normalize("NFC", texto)
    texto = texto.replace("­", "")
    texto = texto.replace(" ", " ").replace("\t", " ")
    texto = re.sub(r"-{2,}", "—", texto)
    return re.sub(r"\s+", " ", texto).strip()


def paragrafos(texto):
    """Parágrafos separados por linha em branco, já sem quebras internas."""
    texto = texto.replace("\r", "")
    return [normaliza(p) for p in re.split(r"\n[ \t ]*\n", texto) if p.strip()]


def chave_indice(p):
    """Forma canônica de uma entrada de índice, para achar sua repetição."""
    return re.sub(r"[\s.:;,!?-]+$", "", p.upper()).strip()


def eh_indice(p):
    return chave_indice(p) in {"ÍNDICE", "INDICE"}


def remove_indices(pars, log):
    """Remove sumários ("ÍNDICE" seguido de entradas curtas)."""
    saida, i = [], 0
    while i < len(pars):
        if not eh_indice(pars[i]):
            saida.append(pars[i])
            i += 1
            continue
        j, entradas, achou = i + 1, [], False
        while j < len(pars) and len(pars[j]) <= 150:
            chave = chave_indice(pars[j])
            if entradas:
                primeira = chave_indice(entradas[0])
                if (chave == primeira or chave.startswith(primeira + " ")
                        or (len(chave) >= 8 and primeira.startswith(chave + " "))):
                    achou = True
                    break
            if eh_indice(pars[j]):
                break
            entradas.append(pars[j])
            j += 1
        log.append({"entradas": len(entradas), "repeticao": achou,
                    "primeira": entradas[0] if entradas else ""})
        i = j
    return saida


def le_obra(caminho):
    rel = caminho.relative_to(ACERVO).as_posix()
    bruto = caminho.read_bytes().decode("cp1252")
    pars = paragrafos(bruto)

    anos = re.findall(r"1[89]\d\d", pars[0])
    ano = int(anos[0]) if anos else None
    titulo = pars[1]

    if rel in CABECALHO_MANUAL:
        inicio = CABECALHO_MANUAL[rel]
    else:
        pub = [i for i, p in enumerate(pars[:13]) if re.match(r"Publicad[oa]s?\b", p)]
        if not pub:
            raise ValueError(f"{rel}: cabeçalho não reconhecido")
        inicio = pub[0] + 1

    log = []
    corpo = remove_indices(pars[inicio:], log)
    return {
        "arquivo": rel,
        "genero": GENEROS[rel.split("/")[0]],
        "titulo": titulo,
        "ano": ano,
        "paragrafos": [titulo] + corpo,
        "indices": log,
    }


def hash_par(p):
    return hashlib.sha1(p.encode("utf-8")).hexdigest()


def constroi(incluir_traducoes=False, saida=SAIDA):
    arquivos = sorted(ACERVO.glob("*/*.txt"))
    obras = [le_obra(a) for a in arquivos]
    if not incluir_traducoes:
        obras = [o for o in obras if o["genero"] != "tradução"]

    vistos, duplicatas = {}, []
    for o in obras:
        mantidos = []
        for p in o["paragrafos"]:
            if len(p) >= MIN_DUPLICATA:
                h = hash_par(p)
                if h in vistos:
                    duplicatas.append({"arquivo": o["arquivo"], "original": vistos[h],
                                       "inicio": p[:80]})
                    continue
                vistos[h] = o["arquivo"]
            mantidos.append(p)
        o["paragrafos"] = mantidos

    splits = {"train": [], "val": [], "test": []}
    g = 0
    for o in obras:
        o["chars"] = {"train": 0, "val": 0, "test": 0}
        bloco = []
        tamanho = 0
        blocos = []
        for p in o["paragrafos"]:
            bloco.append(p)
            tamanho += len(p) + 1
            if tamanho >= TAM_BLOCO:
                blocos.append(bloco)
                bloco, tamanho = [], 0
        if bloco:
            blocos.append(bloco)
        for b in blocos:
            r = g % CICLO
            nome = "val" if r == CICLO - 2 else "test" if r == CICLO - 1 else "train"
            texto = "\n".join(b)
            splits[nome].append((o["arquivo"], texto))
            o["chars"][nome] += len(texto) + 1
            g += 1

    saida.mkdir(parents=True, exist_ok=True)
    completo = "\n\n".join("\n".join(o["paragrafos"]) for o in obras) + "\n"
    (saida / "machado_obra_completa.txt").write_text(completo, encoding="utf-8")

    for nome, blocos in splits.items():
        partes, anterior = [], None
        for arq, texto in blocos:
            if anterior is not None:
                partes.append("\n\n" if arq != anterior else "\n")
            partes.append(texto)
            anterior = arq
        (saida / f"{nome}.txt").write_text("".join(partes) + "\n", encoding="utf-8")

    with open(saida / "obras.jsonl", "w", encoding="utf-8") as f:
        for o in obras:
            f.write(json.dumps({k: o[k] for k in ("arquivo", "genero", "titulo", "ano", "paragrafos")},
                               ensure_ascii=False) + "\n")

    genero_de = {o["arquivo"]: o["genero"] for o in obras}
    por_gen = {}
    for arq, texto in splits["test"]:
        por_gen.setdefault(genero_de[arq], []).append(texto)
    (saida / "test_por_genero").mkdir(exist_ok=True)
    for gen, textos_gen in por_gen.items():
        (saida / "test_por_genero" / f"{gen}.txt").write_text(
            "\n\n".join(textos_gen) + "\n", encoding="utf-8")

    with open(saida / "manifesto.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["arquivo", "genero", "titulo", "ano", "paragrafos", "caracteres",
                    "chars_treino", "chars_val", "chars_teste"])
        for o in obras:
            w.writerow([o["arquivo"], o["genero"], o["titulo"], o["ano"], len(o["paragrafos"]),
                        sum(len(p) + 1 for p in o["paragrafos"]),
                        o["chars"]["train"], o["chars"]["val"], o["chars"]["test"]])

    textos = {n: (saida / f"{n}.txt").read_text(encoding="utf-8") for n in splits}
    freq = Counter(completo)
    por_genero = Counter()
    for o in obras:
        por_genero[o["genero"]] += sum(len(p) + 1 for p in o["paragrafos"])
    stats = {
        "obras": len(obras),
        "incluir_traducoes": incluir_traducoes,
        "caracteres_total": len(completo),
        "palavras_total": len(re.findall(r"\w+", completo)),
        "caracteres_por_split": {n: len(t) for n, t in textos.items()},
        "caracteres_por_genero": dict(por_genero.most_common()),
        "vocabulario_chars": len(freq),
        "chars_raros": {c: n for c, n in freq.items() if n < 5},
        "duplicatas_removidas": len(duplicatas),
        "indices_removidos": sum(len(o["indices"]) for o in obras),
        "indices_sem_repeticao": [
            {"arquivo": o["arquivo"], **l} for o in obras for l in o["indices"] if not l["repeticao"]
        ],
    }
    (saida / "estatisticas.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2),
                                             encoding="utf-8")
    (saida / "duplicatas.json").write_text(json.dumps(duplicatas, ensure_ascii=False, indent=2),
                                           encoding="utf-8")
    return stats


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--incluir-traducoes", action="store_true")
    ap.add_argument("--saida", type=Path, default=SAIDA)
    args = ap.parse_args()
    s = constroi(args.incluir_traducoes, args.saida)
    resumo = {k: v for k, v in s.items() if k not in ("chars_raros", "indices_sem_repeticao")}
    print(json.dumps(resumo, ensure_ascii=False, indent=2))
    print(f"índices sem repetição (revisar): {len(s['indices_sem_repeticao'])}")


if __name__ == "__main__":
    main()

Writing machado_gpt/corpus.py


In [7]:
# [AUTORIA: ALUNO]
!python -m machado_gpt.corpus
texto = open("corpus/machado_obra_completa.txt", encoding="utf-8").read()
i = texto.find("Dom Casmurro\nCAPÍTULO PRIMEIRO")
print(texto[i:i + 800])

{
  "obras": 243,
  "incluir_traducoes": false,
  "caracteres_total": 13393592,
  "palavras_total": 2360804,
  "caracteres_por_split": {
    "train": 12040545,
    "val": 676052,
    "test": 677183
  },
  "caracteres_por_genero": {
    "conto": 5262848,
    "crônica": 3097864,
    "romance": 3053817,
    "crítica": 898553,
    "poesia": 651150,
    "teatro": 361456,
    "miscelânea": 67662
  },
  "vocabulario_chars": 144,
  "duplicatas_removidas": 123,
  "indices_removidos": 45
}
índices sem repetição (revisar): 5
Dom Casmurro
CAPÍTULO PRIMEIRO
DO TÍTULO
Uma noite destas, vindo da cidade para o Engenho Novo, encontrei no trem da Central um rapaz aqui do bairro, que eu conheço de vista e de chapéu. Cumprimentou-me, sentou-se ao pé de mim, falou da Lua e dos ministros, e acabou recitando-me versos. A viagem era curta, e os versos pode ser que não fossem inteiramente maus. Sucedeu, porém, que, como eu estava cansado, fechei os olhos três ou quatro vezes; tanto bastou para que ele interrom

## 4. Tokenização

In [8]:
%%writefile machado_gpt/tokenizacao.py
# [AUTORIA: ALUNO]
"""Tokenizadores usados nos experimentos e preparação dos .bin do nanoGPT."""

import argparse
import json
import pickle
import sys
from pathlib import Path

import numpy as np

RAIZ = Path(__file__).resolve().parent.parent
CORPUS = RAIZ / "corpus"
NANOGPT = RAIZ / "nanoGPT-master"
HF_PTBR = "pierreguillou/gpt2-small-portuguese"

DATASETS = {"char": "machado_char", "gpt2": "machado_gpt2", "ptbr": "machado_ptbr"}


class TokChar:
    def __init__(self, stoi):
        self.stoi = stoi
        self.itos = {i: c for c, i in stoi.items()}
        self.vocab_size = len(stoi)

    @classmethod
    def do_texto(cls, texto):
        return cls({c: i for i, c in enumerate(sorted(set(texto)))})

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)


class TokTiktoken:
    def __init__(self, nome="gpt2"):
        import tiktoken
        self.enc = tiktoken.get_encoding(nome)
        self.vocab_size = self.enc.n_vocab
        self.eot = self.enc.eot_token

    def encode(self, s):
        return self.enc.encode_ordinary(s)

    def decode(self, ids):
        return self.enc.decode(ids)


class TokHF:
    def __init__(self, nome=HF_PTBR):
        from transformers import AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(nome)
        self.vocab_size = len(self.tok)

    def encode(self, s):
        linhas = s.splitlines(keepends=True)
        ids = []
        for i in range(0, len(linhas), 20000):
            for seq in self.tok(linhas[i:i + 20000], add_special_tokens=False)["input_ids"]:
                ids.extend(seq)
        return ids

    def decode(self, ids):
        return self.tok.decode(ids, clean_up_tokenization_spaces=False)


def carrega(data_dir):
    """Reconstrói o tokenizador de um diretório de dados."""
    data_dir = Path(data_dir)
    info = json.loads((data_dir / "tokenizador.json").read_text())
    if info["tipo"] == "char":
        with open(data_dir / "meta.pkl", "rb") as f:
            return TokChar(pickle.load(f)["stoi"])
    if info["tipo"] == "tiktoken":
        return TokTiktoken(info["nome"])
    return TokHF(info["nome"])


def prepara(tipo, corpus=CORPUS, destino=None):
    destino = Path(destino or NANOGPT / "data" / DATASETS[tipo])
    destino.mkdir(parents=True, exist_ok=True)
    textos = {n: (corpus / f"{n}.txt").read_text(encoding="utf-8") for n in ("train", "val", "test")}

    if tipo == "char":
        tok = TokChar.do_texto("".join(textos.values()))
        info_tok = {"tipo": "char"}
        with open(destino / "meta.pkl", "wb") as f:
            pickle.dump({"vocab_size": tok.vocab_size, "itos": tok.itos, "stoi": tok.stoi}, f)
    elif tipo == "gpt2":
        tok = TokTiktoken("gpt2")
        info_tok = {"tipo": "tiktoken", "nome": "gpt2"}
    else:
        tok = TokHF(HF_PTBR)
        info_tok = {"tipo": "hf", "nome": HF_PTBR}
    assert tok.vocab_size < 2 ** 16, "os ids precisam caber em uint16"

    info = {"tokenizador": info_tok, "vocab_size": tok.vocab_size}
    for nome, texto in textos.items():
        ids = tok.encode(texto)
        assert tok.decode(ids) == texto, f"{tipo}: decode(encode(x)) != x em {nome}"
        np.array(ids, dtype=np.uint16).tofile(destino / f"{nome}.bin")
        info[nome] = {"tokens": len(ids), "caracteres": len(texto),
                      "caracteres_por_token": round(len(texto) / len(ids), 4)}
        print(f"[{tipo}] {nome}: {len(texto):,} caracteres -> {len(ids):,} tokens")

    (destino / "tokenizador.json").write_text(json.dumps(info_tok, ensure_ascii=False))
    (destino / "info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2))
    return info


def main():
    ap = argparse.ArgumentParser(description="Gera os .bin do nanoGPT para cada tokenizador.")
    ap.add_argument("tipos", nargs="+", choices=sorted(DATASETS))
    args = ap.parse_args()
    for t in args.tipos:
        prepara(t)


if __name__ == "__main__":
    sys.exit(main())

Writing machado_gpt/tokenizacao.py


In [9]:
# [AUTORIA: ALUNO]
!python -m machado_gpt.tokenizacao char gpt2 ptbr

[char] train: 12,040,545 caracteres -> 12,040,545 tokens
[char] val: 676,052 caracteres -> 676,052 tokens
[char] test: 677,183 caracteres -> 677,183 tokens
[gpt2] train: 12,040,545 caracteres -> 4,757,013 tokens
[gpt2] val: 676,052 caracteres -> 266,291 tokens
[gpt2] test: 677,183 caracteres -> 267,254 tokens
config.json: 100% 666/666 [00:00<00:00, 3.99MB/s]
tokenizer_config.json: 100% 92.0/92.0 [00:00<00:00, 622kB/s]
vocab.json: 100% 850k/850k [00:00<00:00, 45.9MB/s]
merges.txt: 100% 508k/508k [00:00<00:00, 104MB/s]
special_tokens_map.json: 100% 120/120 [00:00<00:00, 868kB/s]
[ptbr] train: 12,040,545 caracteres -> 3,001,003 tokens
[ptbr] val: 676,052 caracteres -> 167,860 tokens
[ptbr] test: 677,183 caracteres -> 168,251 tokens


## 5. Avaliação (bits por caractere) e linhas de base

In [10]:
%%writefile machado_gpt/avaliacao.py
# [AUTORIA: ALUNO]
"""Avaliação comparável entre tokenizadores: bits por caractere (bpc)."""

import argparse
import json
import math
import re
import sys
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
from torch.nn import functional as F

RAIZ = Path(__file__).resolve().parent.parent
NANOGPT = RAIZ / "nanoGPT-master"
CORPUS = RAIZ / "corpus"
sys.path.insert(0, str(NANOGPT))

from model import GPT, GPTConfig

from machado_gpt import tokenizacao


class Modelo:
    """Envelope comum para modelos do nanoGPT, do vídeo e checkpoints do HF."""

    def __init__(self, rede, block_size, dataset, tipo, nome):
        self.rede, self.block_size, self.dataset = rede, block_size, dataset
        self.tipo, self.nome = tipo, nome

    def logits(self, x):
        if self.tipo == "video":
            return self.rede(x)[0]
        return self.rede(x, torch.zeros_like(x))[0]

    @torch.no_grad()
    def gera(self, idx, max_new_tokens, temperature=0.8, top_k=200):
        if self.tipo == "video":
            from machado_gpt import video_gpt
            return video_gpt.gera_com_amostragem(self.rede, idx, max_new_tokens, temperature, top_k)
        return self.rede.generate(idx, max_new_tokens, temperature=temperature, top_k=top_k)

    def n_params(self):
        return sum(p.numel() for p in self.rede.parameters())


def carrega_run(run_dir, device="cpu"):
    """Carrega o checkpoint de um run (nanoGPT ou vídeo)."""
    run_dir = Path(run_dir)
    ck_path = run_dir / "ckpt.pt"
    if not ck_path.exists():
        ck_path = run_dir / "ckpt_slim.pt"
    ck = torch.load(ck_path, map_location=device, weights_only=False)
    if ck.get("tipo") == "video":
        from machado_gpt import video_gpt
        video_gpt.configura(**ck["hparams"], device=device)
        rede = video_gpt.GPTLanguageModel()
        rede.load_state_dict({k: v.float() for k, v in ck["model"].items()})
        rede.to(device).eval()
        return Modelo(rede, ck["hparams"]["block_size"], ck["dataset"], "video", run_dir.name)
    rede = GPT(GPTConfig(**ck["model_args"]))
    sd = {k.removeprefix("_orig_mod."): v.float() for k, v in ck["model"].items()}
    rede.load_state_dict(sd)
    rede.to(device).eval()
    return Modelo(rede, ck["model_args"]["block_size"], ck["config"]["dataset"], "nanogpt", run_dir.name)


def carrega_hf(hf_id, dataset, block_size=1024, device="cpu"):
    rede = GPT.from_pretrained(hf_id, dict(dropout=0.0))
    if block_size < rede.config.block_size:
        rede.crop_block_size(block_size)
    rede.to(device).eval()
    return Modelo(rede, rede.config.block_size, dataset, "nanogpt", hf_id)


@torch.no_grad()
def nll_total(modelo, ids, device="cpu", passo=None, lote=16, ctx=nullcontext()):
    """Soma das NLL (nats) de ids[1:], cada token pontuado uma vez."""
    T = modelo.block_size
    passo = passo or T // 2
    ids = torch.as_tensor(np.asarray(ids, dtype=np.int64))
    N = len(ids)
    janelas = []
    fim_anterior = 0
    fim = min(T, N - 1)
    while True:
        inicio = max(0, fim - T)
        janelas.append((inicio, fim, fim - fim_anterior))
        fim_anterior = fim
        if fim == N - 1:
            break
        fim = min(fim + passo, N - 1)

    soma, contagem = 0.0, 0
    for i in range(0, len(janelas), lote):
        grupo = janelas[i:i + lote]
        comp = max(f - s for s, f, _ in grupo)
        x = torch.zeros((len(grupo), comp), dtype=torch.long)
        y = torch.full((len(grupo), comp), -1, dtype=torch.long)
        for b, (s, f, novos) in enumerate(grupo):
            n = f - s
            x[b, :n] = ids[s:f]
            y[b, n - novos:n] = ids[f - novos + 1:f + 1]
        x, y = x.to(device), y.to(device)
        with ctx:
            logits = modelo.logits(x)
        perdas = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), y.reshape(-1),
                                 ignore_index=-1, reduction="sum")
        soma += perdas.item()
        contagem += int((y >= 0).sum().item())
    assert contagem == N - 1
    return soma, contagem


def avalia_texto(modelo, tok, texto, device="cpu", ctx=nullcontext(), lote=16):
    ids = tok.encode(texto)
    t0 = time.time()
    soma, n = nll_total(modelo, ids, device=device, ctx=ctx, lote=lote)
    n_chars = len(texto) - len(tok.decode(ids[:1]))
    n_palavras = len(re.findall(r"\w+", texto))
    return {
        "tokens": n,
        "caracteres": n_chars,
        "perda_por_token": soma / n,
        "bpc": soma / (n_chars * math.log(2)),
        "ppl_palavra": math.exp(soma / n_palavras),
        "segundos": round(time.time() - t0, 1),
    }


def avalia(modelo, device="cpu", splits=("val", "test"), por_genero=True, lote=16, limite_chars=None):
    """Avalia em val/test e no teste por gênero. `limite_chars` corta os textos (teste rápido)."""
    data_dir = NANOGPT / "data" / modelo.dataset
    tok = tokenizacao.carrega(data_dir)
    ctx = (torch.amp.autocast(device_type="cuda", dtype=torch.float16)
           if "cuda" in str(device) else nullcontext())
    res = {"modelo": modelo.nome, "dataset": modelo.dataset, "parametros": modelo.n_params(),
           "block_size": modelo.block_size}
    for s in splits:
        texto = (CORPUS / f"{s}.txt").read_text(encoding="utf-8")[:limite_chars]
        res[s] = avalia_texto(modelo, tok, texto, device, ctx, lote)
        print(f"[{modelo.nome}] {s}: bpc={res[s]['bpc']:.4f} ppl_palavra={res[s]['ppl_palavra']:.1f}",
              flush=True)
    if por_genero:
        res["test_por_genero"] = {}
        for arq in sorted((CORPUS / "test_por_genero").glob("*.txt")):
            texto = arq.read_text(encoding="utf-8")[:limite_chars]
            r = avalia_texto(modelo, tok, texto, device, ctx, lote)
            res["test_por_genero"][arq.stem] = {"bpc": r["bpc"], "caracteres": r["caracteres"]}
    return res


def main():
    ap = argparse.ArgumentParser(description="Avalia um modelo em bits por caractere.")
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--run", help="diretório do run com ckpt.pt")
    g.add_argument("--hf", help="id de checkpoint GPT-2 do Hugging Face (zero-shot)")
    ap.add_argument("--dataset", help="dataset do tokenizador (obrigatório com --hf)")
    ap.add_argument("--block-size", type=int, default=1024)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--lote", type=int, default=16)
    ap.add_argument("--saida", help="json de saída")
    ap.add_argument("--limite-chars", type=int, default=None,
                    help="avalia só os primeiros N caracteres de cada texto (estimativa rápida)")
    args = ap.parse_args()

    if args.run:
        modelo = carrega_run(args.run, args.device)
    else:
        modelo = carrega_hf(args.hf, args.dataset, args.block_size, args.device)
    res = avalia(modelo, args.device, lote=args.lote, limite_chars=args.limite_chars)
    txt = json.dumps(res, ensure_ascii=False, indent=2)
    if args.saida:
        Path(args.saida).parent.mkdir(parents=True, exist_ok=True)
        Path(args.saida).write_text(txt, encoding="utf-8")
    print(txt)


if __name__ == "__main__":
    main()

Writing machado_gpt/avaliacao.py


In [11]:
%%writefile machado_gpt/baselines.py
# [AUTORIA: ALUNO]
"""Linhas de base de n-gramas de caracteres (unigrama, bigrama, trigrama)."""

import json
import math
from pathlib import Path

import numpy as np

RAIZ = Path(__file__).resolve().parent.parent
DATA = RAIZ / "nanoGPT-master" / "data" / "machado_char"
ALFAS = [1e-3, 1e-2, 0.1, 0.5, 1.0]


def contagens(ids, n, V):
    tab = np.zeros((V,) * n, dtype=np.float64)
    idx = tuple(ids[i:len(ids) - n + 1 + i] for i in range(n))
    np.add.at(tab, idx, 1)
    return tab


def bpc(tab, ids, n, alfa):
    V = tab.shape[-1]
    probs = (tab + alfa) / (tab.sum(axis=-1, keepdims=True) + alfa * V)
    idx = tuple(ids[i:len(ids) - n + 1 + i] for i in range(n))
    return float(-np.log2(probs[idx]).mean())


def main():
    V = json.loads((DATA / "info.json").read_text())["vocab_size"]
    ids = {s: np.fromfile(DATA / f"{s}.bin", dtype=np.uint16).astype(np.int64)
           for s in ("train", "val", "test")}
    res = {}
    for n, nome in [(1, "unigrama"), (2, "bigrama"), (3, "trigrama")]:
        tab = contagens(ids["train"], n, V)
        alfa = min(ALFAS, key=lambda a: bpc(tab, ids["val"], n, a))
        res[nome] = {
            "alfa": alfa,
            "parametros": V ** n,
            "val": {"bpc": bpc(tab, ids["val"], n, alfa)},
            "test": {"bpc": bpc(tab, ids["test"], n, alfa)},
        }
        res[nome]["test"]["perda_por_token"] = res[nome]["test"]["bpc"] * math.log(2)
        print(f"{nome}: alfa={alfa} val={res[nome]['val']['bpc']:.4f} "
              f"test={res[nome]['test']['bpc']:.4f} bpc")
    saida = RAIZ / "results" / "metrics" / "baselines_ngrama.json"
    saida.parent.mkdir(parents=True, exist_ok=True)
    saida.write_text(json.dumps(res, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

Writing machado_gpt/baselines.py


In [12]:
# [AUTORIA: ALUNO]
!python -m machado_gpt.baselines

unigrama: alfa=1.0 val=4.4789 test=4.4700 bpc
bigrama: alfa=0.1 val=3.4212 test=3.4224 bpc
trigrama: alfa=0.01 val=2.8578 test=2.8599 bpc


## 6. Modelos e treino

In [13]:
%%writefile machado_gpt/video_gpt.py
# [AUTORIA: TERCEIROS] gpt.py do vídeo, Andrej Karpathy (MIT); alterações do aluno marcadas com # [ALUNO]

import argparse
import json
import pickle
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters (valores do vídeo, gpt.py)
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
vocab_size = None  # [ALUNO] definido pelos dados
# ------------

RAIZ = Path(__file__).resolve().parent.parent  # [ALUNO]
DATA = RAIZ / "nanoGPT-master" / "data" / "machado_char"  # [ALUNO]


def configura(**kw):  # [ALUNO] permite reconstruir o modelo fora deste script
    g = globals()
    for k, v in kw.items():
        assert k in g, k
        g[k] = v


class Head(nn.Module):
    """one head of self-attention"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """multiple heads of self-attention in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """a simple linear layer followed by a non-linearity"""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """Transformer block: communication followed by computation"""

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T,C)  [ALUNO] device do tensor de entrada
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


@torch.no_grad()
def gera_com_amostragem(model, idx, max_new_tokens, temperature=0.8, top_k=200):
    """[ALUNO] generate() do vídeo + temperatura e top-k, como no nanoGPT."""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -block_size:])
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')
        probs = F.softmax(logits, dim=-1)
        idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
    return idx


def main():
    ap =argparse.ArgumentParser(description="Treina o GPT do vídeo no corpus de Machado.")  # [ALUNO]
    ap.add_argument("--out", default=str(RAIZ / "runs" / "video_gpt"))
    ap.add_argument("--max-iters", type=int, default=max_iters)
    ap.add_argument("--eval-iters", type=int, default=eval_iters)
    ap.add_argument("--device", default=device)
    for nome in ("batch_size", "block_size", "n_embd", "n_head", "n_layer"):  # tamanho (teste rápido)
        ap.add_argument("--" + nome.replace("_", "-"), type=int, default=globals()[nome])
    args = ap.parse_args()
    configura(max_iters=args.max_iters, eval_iters=args.eval_iters, device=args.device,
              batch_size=args.batch_size, block_size=args.block_size, n_embd=args.n_embd,
              n_head=args.n_head, n_layer=args.n_layer)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(1337)

    # [ALUNO] dados: mesmos .bin e vocabulário do experimento char do nanoGPT
    with open(DATA / "meta.pkl", "rb") as f:
        meta = pickle.load(f)
    configura(vocab_size=meta["vocab_size"])
    decode = lambda l: ''.join([meta["itos"][i] for i in l])
    train_data = torch.from_numpy(np.fromfile(DATA / "train.bin", dtype=np.uint16).astype(np.int64))
    val_data = torch.from_numpy(np.fromfile(DATA / "val.bin", dtype=np.uint16).astype(np.int64))

    # data loading
    def get_batch(split):
        # generate a small batch of data of inputs x and targets y
        data = train_data if split == 'train' else val_data
        ix = torch.randint(len(data) - block_size, (batch_size,))
        x = torch.stack([data[i:i+block_size] for i in ix])
        y = torch.stack([data[i+1:i+block_size+1] for i in ix])
        x, y = x.to(device), y.to(device)
        return x, y

    @torch.no_grad()
    def estimate_loss():
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
        model.train()
        return out

    model = GPTLanguageModel()
    m = model.to(device)
    # print the number of parameters in the model
    print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

    # create a PyTorch optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    hparams = dict(batch_size=batch_size, block_size=block_size, learning_rate=learning_rate,
                   n_embd=n_embd, n_head=n_head, n_layer=n_layer, dropout=dropout,
                   vocab_size=vocab_size, max_iters=max_iters)  # [ALUNO]
    best_val, t0, log = float("inf"), time.time(), []  # [ALUNO]
    for iter in range(max_iters):

        # every once in a while evaluate the loss on train and val sets
        if iter % eval_interval == 0 or iter == max_iters - 1:
            losses = estimate_loss()
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}", flush=True)
            # [ALUNO] log e checkpoint do melhor modelo na validação
            dt = time.time() - t0
            log.append({"iter": iter, "train": float(losses['train']), "val": float(losses['val']),
                        "segundos": round(dt, 1)})
            if losses['val'] < best_val:
                best_val = float(losses['val'])
                torch.save({"tipo": "video", "hparams": hparams, "dataset": "machado_char",
                            "model": model.state_dict(), "iter_num": iter, "best_val_loss": best_val},
                           out / "ckpt.pt")

        # sample a batch of data
        xb, yb = get_batch('train')

        # evaluate the loss
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # [ALUNO] resumo do treino
    total = time.time() - t0
    (out / "treino.json").write_text(json.dumps(
        {"log": log, "segundos_total": round(total, 1), "ms_por_iter": round(1000 * total / max_iters, 1),
         "best_val_loss": best_val, "hparams": hparams}, indent=2))

    # generate from the model
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


if __name__ == "__main__":
    main()

Writing machado_gpt/video_gpt.py


In [14]:
%%writefile experiments/configs/_comum.py
# [AUTORIA: ALUNO]
device = 'cuda'
dtype = 'float16'
compile = True
wandb_log = False
always_save_checkpoint = False
log_interval = 50

Writing experiments/configs/_comum.py


In [15]:
%%writefile experiments/configs/char_baby.py
# [AUTORIA: ALUNO]
exec(open('../experiments/configs/_comum.py').read())
out_dir = '../runs/char_baby'
dataset = 'machado_char'
eval_interval = 250
eval_iters = 200
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2
learning_rate = 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100

Writing experiments/configs/char_baby.py


In [16]:
%%writefile experiments/configs/char_cpu.py
# [AUTORIA: ALUNO]
exec(open('../experiments/configs/_comum.py').read())
out_dir = '../runs/char_cpu'
dataset = 'machado_char'
device = 'cpu'
compile = False
dtype = 'float32'
eval_interval = 250
eval_iters = 20
log_interval = 50
gradient_accumulation_steps = 1
batch_size = 12
block_size = 64
n_layer = 4
n_head = 4
n_embd = 128
dropout = 0.0
learning_rate = 1e-3
max_iters = 2000
lr_decay_iters = 2000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100

Writing experiments/configs/char_cpu.py


In [17]:
%%writefile experiments/configs/char_large.py
# [AUTORIA: ALUNO]
exec(open('../experiments/configs/_comum.py').read())
out_dir = '../runs/char_large'
dataset = 'machado_char'
eval_interval = 250
eval_iters = 100
gradient_accumulation_steps = 1
batch_size = 32
block_size = 512
n_layer = 8
n_head = 8
n_embd = 512
dropout = 0.2
learning_rate = 6e-4
max_iters = 6000
lr_decay_iters = 6000
min_lr = 6e-5
beta2 = 0.99
warmup_iters = 200

Writing experiments/configs/char_large.py


In [18]:
%%writefile experiments/configs/gpt2_ft.py
# [AUTORIA: ALUNO]
exec(open('../experiments/configs/_comum.py').read())
out_dir = '../runs/gpt2_ft'
dataset = 'machado_gpt2'
init_from = 'gpt2'
eval_interval = 100
eval_iters = 40
log_interval = 10
batch_size = 8
gradient_accumulation_steps = 4
block_size = 512
dropout = 0.1
learning_rate = 6e-5
max_iters = 1200
lr_decay_iters = 1200
min_lr = 6e-6
warmup_iters = 50
weight_decay = 0.1

Writing experiments/configs/gpt2_ft.py


In [19]:
%%writefile experiments/configs/gpt2_scratch.py
# [AUTORIA: ALUNO]
exec(open('../experiments/configs/_comum.py').read())
out_dir = '../runs/gpt2_scratch'
dataset = 'machado_gpt2'
eval_interval = 250
eval_iters = 100
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2
learning_rate = 1e-3
max_iters = 3000
lr_decay_iters = 3000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100

Writing experiments/configs/gpt2_scratch.py


In [20]:
%%writefile experiments/configs/ptbr_ft.py
# [AUTORIA: ALUNO]
exec(open('../experiments/configs/_comum.py').read())
out_dir = '../runs/ptbr_ft'
dataset = 'machado_ptbr'
init_from = 'pierreguillou/gpt2-small-portuguese'
eval_interval = 100
eval_iters = 40
log_interval = 10
batch_size = 8
gradient_accumulation_steps = 4
block_size = 512
dropout = 0.1
learning_rate = 6e-5
max_iters = 1200
lr_decay_iters = 1200
min_lr = 6e-6
warmup_iters = 50
weight_decay = 0.1

Writing experiments/configs/ptbr_ft.py


In [21]:
%%writefile experiments/configs/ptbr_scratch.py
# [AUTORIA: ALUNO]
exec(open('../experiments/configs/_comum.py').read())
out_dir = '../runs/ptbr_scratch'
dataset = 'machado_ptbr'
eval_interval = 250
eval_iters = 100
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2
learning_rate = 1e-3
max_iters = 3000
lr_decay_iters = 3000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100

Writing experiments/configs/ptbr_scratch.py


In [22]:
%%writefile machado_gpt/geracao.py
# [AUTORIA: ALUNO]
"""Geração de amostras e métricas de qualidade do texto gerado."""

import argparse
import json
import math
import random
import re
from pathlib import Path

import torch

from machado_gpt import tokenizacao
from machado_gpt.avaliacao import NANOGPT, CORPUS, RAIZ, carrega_hf, carrega_run

N_PROMPTS, AMOSTRAS_POR_PROMPT, CHARS_PROMPT, CHARS_GERADOS = 10, 3, 150, 400
SEMENTE = 1337

VITRINE = [
    "Capitu olhou para mim",
    "— Não, senhor, disse ele;",
    "A vida é uma ópera,",
]


def palavras(texto):
    return re.findall(r"[^\W\d_]+", texto.lower())


def prompts_do_teste(n=N_PROMPTS, semente=SEMENTE):
    pars = [p for p in (CORPUS / "test.txt").read_text(encoding="utf-8").split("\n")
            if len(p) >= CHARS_PROMPT + CHARS_GERADOS + 50]
    rng = random.Random(semente)
    escolhidos = rng.sample(pars, n)
    saida = []
    for p in escolhidos:
        corte = p.rfind(" ", 0, CHARS_PROMPT)
        saida.append({"prompt": p[:corte], "referencia": p[corte:corte + CHARS_GERADOS]})
    return saida


class Metricas:
    """Métricas de texto; carrega o vocabulário e os 8-gramas do treino uma vez."""

    def __init__(self):
        treino = (CORPUS / "train.txt").read_text(encoding="utf-8")
        ws = palavras(treino)
        self.vocab = set(ws)
        self.oito = {hash(tuple(ws[i:i + 8])) for i in range(len(ws) - 7)}

    def calcula(self, textos):
        todas, uni, bi, oito_tot, oito_mem, rep_tot, rep = [], set(), set(), 0, 0, 0, 0
        n_bi = 0
        for t in textos:
            ws = palavras(t)
            todas.extend(ws)
            uni.update(ws)
            bigr = [tuple(ws[i:i + 2]) for i in range(len(ws) - 1)]
            bi.update(bigr)
            n_bi += len(bigr)
            g8 = [hash(tuple(ws[i:i + 8])) for i in range(len(ws) - 7)]
            oito_tot += len(g8)
            oito_mem += sum(h in self.oito for h in g8)
            g4 = [tuple(ws[i:i + 4]) for i in range(len(ws) - 3)]
            rep_tot += len(g4)
            rep += len(g4) - len(set(g4))
        return {
            "palavras": len(todas),
            "palavras_validas": sum(w in self.vocab for w in todas) / max(1, len(todas)),
            "distinct_1": len(uni) / max(1, len(todas)),
            "distinct_2": len(bi) / max(1, n_bi),
            "memorizacao_8grama": oito_mem / max(1, oito_tot),
            "repeticao_4grama": rep / max(1, rep_tot),
        }


def gera_texto(modelo, tok, prompt, n_chars, device, ratio):
    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
    n_tokens = math.ceil(n_chars / ratio * 1.3) + 8
    ctx = (torch.amp.autocast(device_type="cuda", dtype=torch.float16)
           if "cuda" in str(device) else torch.no_grad())
    with ctx:
        y = modelo.gera(ids, n_tokens)
    texto = tok.decode(y[0].tolist())
    return texto[len(prompt):][:n_chars]


def roda(modelo, device, saida_dir, n_prompts=N_PROMPTS, por_prompt=AMOSTRAS_POR_PROMPT,
         n_chars=CHARS_GERADOS):
    data_dir = NANOGPT / "data" / modelo.dataset
    tok = tokenizacao.carrega(data_dir)
    info = json.loads((data_dir / "info.json").read_text())
    ratio = info["train"]["caracteres_por_token"]
    torch.manual_seed(SEMENTE)

    amostras = []
    for p in prompts_do_teste()[:n_prompts]:
        for k in range(por_prompt):
            amostras.append({"prompt": p["prompt"],
                             "gerado": gera_texto(modelo, tok, p["prompt"], n_chars, device, ratio)})
    vitrine = [{"prompt": v, "gerado": gera_texto(modelo, tok, v, int(1.5 * n_chars), device, ratio)}
               for v in VITRINE]

    metricas = Metricas().calcula([a["gerado"] for a in amostras])
    saida_dir = Path(saida_dir)
    saida_dir.mkdir(parents=True, exist_ok=True)
    (saida_dir / f"{modelo.nome.replace('/', '_')}.json").write_text(json.dumps(
        {"modelo": modelo.nome, "metricas": metricas, "amostras": amostras, "vitrine": vitrine},
        ensure_ascii=False, indent=2), encoding="utf-8")
    return metricas


def referencia(saida_dir):
    ps = prompts_do_teste()
    m = Metricas().calcula([p["referencia"] for p in ps])
    Path(saida_dir).mkdir(parents=True, exist_ok=True)
    (Path(saida_dir) / "referencia_humana.json").write_text(json.dumps(
        {"modelo": "referência humana (teste)", "metricas": m, "amostras": ps},
        ensure_ascii=False, indent=2), encoding="utf-8")
    return m


def main():
    ap = argparse.ArgumentParser(description="Gera amostras e mede a qualidade do texto.")
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--run")
    g.add_argument("--hf")
    g.add_argument("--referencia", action="store_true")
    ap.add_argument("--prompt", help="gera só a continuação deste texto (uso livre do modelo)")
    ap.add_argument("--chars", type=int, default=600, help="tamanho da continuação com --prompt")
    ap.add_argument("--dataset")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--saida", default=str(RAIZ / "results" / "amostras"))
    args = ap.parse_args()
    if args.prompt:
        modelo = (carrega_run(args.run, args.device) if args.run
                  else carrega_hf(args.hf, args.dataset, 1024, args.device))
        data_dir = NANOGPT / "data" / modelo.dataset
        tok = tokenizacao.carrega(data_dir)
        ratio = json.loads((data_dir / "info.json").read_text())["train"]["caracteres_por_token"]
        torch.manual_seed(SEMENTE)
        print(args.prompt + gera_texto(modelo, tok, args.prompt, args.chars, args.device, ratio))
        return
    if args.referencia:
        m = referencia(args.saida)
    else:
        modelo = (carrega_run(args.run, args.device) if args.run
                  else carrega_hf(args.hf, args.dataset, 1024, args.device))
        m = roda(modelo, args.device, args.saida)
    print(json.dumps(m, indent=2))


if __name__ == "__main__":
    main()

Writing machado_gpt/geracao.py


In [23]:
%%writefile experiments/executar.py
# [AUTORIA: ALUNO]
"""Orquestra os experimentos: treino -> avaliação -> geração -> checkpoint enxuto."""

import argparse
import json
import os
import re
import shutil
import statistics
import subprocess
import sys
import time
from pathlib import Path

RAIZ = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(RAIZ))
NANOGPT = RAIZ / "nanoGPT-master"
CONFIGS = RAIZ / "experiments" / "configs"

HF_PTBR = "pierreguillou/gpt2-small-portuguese"
RUNS = {
    "char_cpu": {"tipo": "nanogpt", "config": "char_cpu.py"},
    "video_gpt": {"tipo": "video"},
    "char_baby": {"tipo": "nanogpt", "config": "char_baby.py"},
    "char_large": {"tipo": "nanogpt", "config": "char_large.py"},
    "gpt2_scratch": {"tipo": "nanogpt", "config": "gpt2_scratch.py"},
    "ptbr_scratch": {"tipo": "nanogpt", "config": "ptbr_scratch.py"},
    "gpt2_zeroshot": {"tipo": "hf", "hf": "gpt2", "dataset": "machado_gpt2", "block": 512},
    "ptbr_zeroshot": {"tipo": "hf", "hf": HF_PTBR, "dataset": "machado_ptbr", "block": 512},
    "gpt2_ft": {"tipo": "nanogpt", "config": "gpt2_ft.py"},
    "ptbr_ft": {"tipo": "nanogpt", "config": "ptbr_ft.py"},
}
ORDEM_GPU = ["char_baby", "ptbr_zeroshot", "gpt2_zeroshot", "ptbr_ft", "gpt2_ft",
             "ptbr_scratch", "gpt2_scratch", "char_large", "video_gpt"]

SMOKE = ["--device=cpu", "--compile=False", "--dtype=float32", "--max_iters=6", "--lr_decay_iters=6",
         "--warmup_iters=2", "--eval_interval=3", "--eval_iters=2", "--log_interval=1",
         "--batch_size=2", "--gradient_accumulation_steps=1"]
SMOKE_SCRATCH = ["--n_layer=2", "--n_head=2", "--n_embd=64", "--block_size=64"]
SMOKE_FT = ["--block_size=64"]


class Status:
    def __init__(self, raiz_saida):
        self.arq = raiz_saida / "status.json"
        self.dados = json.loads(self.arq.read_text()) if self.arq.exists() else {}

    def marca(self, run, etapa, **extra):
        self.dados.setdefault(run, {})[etapa] = {"quando": time.strftime("%H:%M:%S"), **extra}
        self.arq.parent.mkdir(parents=True, exist_ok=True)
        self.arq.write_text(json.dumps(self.dados, indent=2, ensure_ascii=False))
        print(f"[executar] {run}: {etapa} {extra if extra else ''}", flush=True)


def le_log_nanogpt(log):
    """Extrai as curvas de perda e o tempo por iteração do stdout do train.py."""
    evals = [{"iter": int(a), "train": float(b), "val": float(c)} for a, b, c in
             re.findall(r"step (\d+): train loss ([\d.]+), val loss ([\d.]+)", log)]
    iters = [{"iter": int(a), "loss": float(b), "ms": float(c)} for a, b, c in
             re.findall(r"iter (\d+): loss ([\d.]+), time ([\d.]+)ms", log)]
    ms = [i["ms"] for i in iters if i["iter"] >= 20] or [i["ms"] for i in iters]
    return {"evals": evals, "iters": iters, "ms_por_iter_mediana": statistics.median(ms) if ms else None}


def treina(run, spec, run_dir, args, status):
    if (run_dir / "treino.json").exists():
        return
    run_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    status.marca(run, "treino_inicio")
    if spec["tipo"] == "video":
        cmd = [sys.executable, "-m", "machado_gpt.video_gpt", "--out", str(run_dir),
               "--device", args.device]
        if args.smoke:
            cmd += ["--max-iters", "6", "--eval-iters", "2", "--batch-size", "2", "--block-size", "64",
                    "--n-embd", "64", "--n-head", "2", "--n-layer", "2"]
        cwd = RAIZ
    else:
        cmd = [sys.executable, "train.py", str(CONFIGS / spec["config"]), f"--out_dir={run_dir}"]
        if args.smoke:
            cmd += SMOKE + (SMOKE_FT if spec["config"].endswith("_ft.py") else SMOKE_SCRATCH)
        elif args.device == "cpu" and spec["config"] != "char_cpu.py":
            cmd += ["--device=cpu", "--compile=False", "--dtype=float32"]
        cwd = NANOGPT
    with open(run_dir / "train.log", "w") as log:
        env = {**os.environ, "PYTHONUNBUFFERED": "1"}
        proc = subprocess.run(cmd, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, env=env)
    texto = (run_dir / "train.log").read_text()
    if proc.returncode != 0:
        status.marca(run, "treino_erro", codigo=proc.returncode)
        raise SystemExit(f"{run}: treino falhou\n{texto[-3000:]}")
    if spec["tipo"] == "video":
        resumo = json.loads((run_dir / "treino.json").read_text())
    else:
        resumo = le_log_nanogpt(texto)
    resumo["segundos_total"] = round(time.time() - t0, 1)
    (run_dir / "treino.json").write_text(json.dumps(resumo, indent=2))
    status.marca(run, "treino_fim", segundos=resumo["segundos_total"])


def enxuga(run_dir):
    """Checkpoint só com pesos em fp16 (sem estado do otimizador), para download."""
    import torch
    ck_path = run_dir / "ckpt.pt"
    if not ck_path.exists() or (run_dir / "ckpt_slim.pt").exists():
        return
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    ck.pop("optimizer", None)
    ck["model"] = {k.removeprefix("_orig_mod."): v.half() for k, v in ck["model"].items()}
    torch.save(ck, run_dir / "ckpt_slim.pt")


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("runs", nargs="*", help=f"runs a executar: {', '.join(RUNS)}")
    ap.add_argument("--todos", action="store_true", help="todos os runs de GPU, na ordem padrão")
    ap.add_argument("--device", default=None)
    ap.add_argument("--smoke", action="store_true", help="teste rápido do pipeline na CPU")
    ap.add_argument("--sem-geracao", action="store_true")
    args = ap.parse_args()

    import torch
    args.device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    if args.smoke:
        args.device = "cpu"
    runs = ORDEM_GPU if args.todos else args.runs
    desconhecidos = [r for r in runs if r not in RUNS]
    if desconhecidos or not runs:
        ap.error(f"runs inválidos ou ausentes: {desconhecidos}; opções: {list(RUNS)}")
    base_runs = RAIZ / ("runs_smoke" if args.smoke else "runs")
    base_res = RAIZ / ("results_smoke" if args.smoke else "results")
    status = Status(base_res)

    from machado_gpt import avaliacao, geracao

    for run in runs:
        spec = RUNS[run]
        run_dir = base_runs / run
        if spec["tipo"] != "hf":
            treina(run, spec, run_dir, args, status)

        met = base_res / "metrics" / f"{run}.json"
        if not met.exists():
            status.marca(run, "avaliacao_inicio")
            if spec["tipo"] == "hf":
                modelo = avaliacao.carrega_hf(spec["hf"], spec["dataset"], spec["block"], args.device)
            else:
                modelo = avaliacao.carrega_run(run_dir, args.device)
            modelo.nome = run
            res = avaliacao.avalia(modelo, args.device, lote=8 if args.device == "cpu" else 16,
                                   limite_chars=3000 if args.smoke else None)
            if spec["tipo"] != "hf":
                res["treino"] = json.loads((run_dir / "treino.json").read_text())
            met.parent.mkdir(parents=True, exist_ok=True)
            met.write_text(json.dumps(res, indent=2, ensure_ascii=False))
            status.marca(run, "avaliacao_fim", bpc_test=round(res["test"]["bpc"], 4))
        else:
            modelo = None

        amostra = base_res / "amostras" / f"{run}.json"
        if not args.sem_geracao and not amostra.exists():
            if modelo is None:
                modelo = (avaliacao.carrega_hf(spec["hf"], spec["dataset"], spec["block"], args.device)
                          if spec["tipo"] == "hf" else avaliacao.carrega_run(run_dir, args.device))
                modelo.nome = run
            status.marca(run, "geracao_inicio")
            kw = dict(n_prompts=2, por_prompt=1, n_chars=60) if args.smoke else {}
            m = geracao.roda(modelo, args.device, base_res / "amostras", **kw)
            status.marca(run, "geracao_fim", **{k: round(v, 4) for k, v in m.items()})
        if spec["tipo"] != "hf":
            enxuga(run_dir)
        del modelo
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not (base_res / "amostras" / "referencia_humana.json").exists():
        geracao.referencia(base_res / "amostras")
    status.marca("_todos", "fim")


if __name__ == "__main__":
    main()

Writing experiments/executar.py


In [ ]:
# [AUTORIA: ALUNO]
EXPERIMENTOS = ["char_cpu", "char_baby", "ptbr_zeroshot", "gpt2_zeroshot", "ptbr_ft", "gpt2_ft", "ptbr_scratch", "gpt2_scratch", "char_large", "video_gpt"]  # ~3 h numa T4; ["char_cpu"] para um teste rápido
!python experiments/executar.py {" ".join(EXPERIMENTOS)}

[executar] char_cpu: treino_inicio 
[executar] char_cpu: treino_fim {'segundos': 154.7}
[executar] char_cpu: avaliacao_inicio 
number of parameters: 0.81M
[char_cpu] val: bpc=2.6938 ppl_palavra=41072.0
[char_cpu] test: bpc=2.6908 ppl_palavra=40260.5
[executar] char_cpu: avaliacao_fim {'bpc_test': 2.6908}
[executar] char_cpu: geracao_inicio 
[executar] char_cpu: geracao_fim {'palavras': 2209, 'palavras_validas': 0.5976, 'distinct_1': 0.5034, 'distinct_2': 0.9385, 'memorizacao_8grama': 0.0, 'repeticao_4grama': 0.0}
[executar] char_baby: treino_inicio 


## 7. Resultados

In [ ]:
# [AUTORIA: ALUNO]
for run in EXPERIMENTOS:
    m = json.load(open(f"results/metrics/{run}.json"))
    print(f"{run:15s} parâmetros={m['parametros']/1e6:7.2f}M  bpc teste={m['test']['bpc']:.4f}")

In [ ]:
# [AUTORIA: ALUNO]
for run in EXPERIMENTOS:
    v = json.load(open(f"results/amostras/{run}.json"))["vitrine"][0]
    print(f"===== {run}\n{v['prompt']}{v['gerado'][:400]}\n")

## 8. Extensão: perguntas com recuperação de passagens

In [ ]:
%%writefile machado_gpt/rag.py
# [AUTORIA: ALUNO]
"""Protótipo de extensão: responder perguntas sobre o universo de Machado."""

import argparse
import json
import re
from pathlib import Path

import numpy as np

RAIZ = Path(__file__).resolve().parent.parent
TAM_PASSAGEM = 600

PERGUNTAS = [
    ("Quem era o melhor amigo de Bentinho no seminário?", "romance/marm08.txt", "Escobar"),
    ("Como se chamava o filho de Capitu e Bentinho?", "romance/marm08.txt", "Ezequiel"),
    ("Onde Bento Santiago mandou construir a casa em que mora?", "romance/marm08.txt", "Engenho Novo"),
    ("A quem Brás Cubas dedica as suas memórias póstumas?", "romance/marm05.txt", "verme"),
    ("Que remédio Brás Cubas queria inventar para aliviar a melancolia da humanidade?", "romance/marm05.txt", "emplasto"),
    ("Qual é a filosofia criada por Quincas Borba?", "romance/marm05.txt", "Humanitismo"),
    ("Por quem Rubião se apaixona no Rio de Janeiro?", "romance/marm07.txt", "Sofia"),
    ("Qual o nome do médico que funda o hospício em O Alienista?", "contos/macn003.txt", "Bacamarte"),
    ("Qual era o posto militar de Jacobina no conto O Espelho?", "contos/macn003.txt", "alferes"),
    ("Quem sente prazer com o sofrimento alheio em A Causa Secreta?", "contos/macn005.txt", "Fortunato"),
    ("Quem vai consultar a cartomante com medo de ser descoberto?", "contos/macn005.txt", "Camilo"),
    ("Com quem Nogueira conversa na noite de Natal em Missa do Galo?", "contos/macn006.txt", "Conceição"),
    ("Qual é o ofício de Cândido Neves em Pai contra mãe?", "contos/macn007.txt", "escravos fugidos"),
    ("Como se chamam os gêmeos rivais filhos de Natividade?", "romance/marm09.txt", "Pedro"),
    ("Quem é a madrasta de Iaiá Garcia?", "romance/marm04.txt", "Estela"),
    ("Quem é o diplomata aposentado que escreve o diário de 1888 e 1889?", "romance/marm10.txt", "Aires"),
    ("Como se chama a filha natural que o Conselheiro Vale reconhece em testamento?", "romance/marm03.txt", "Helena"),
    ("Em que vila Simão Bacamarte abre a Casa Verde?", "contos/macn003.txt", "Itaguaí"),
    ("Que ofício o pai recomenda a Janjão quando ele completa vinte e um anos?", "contos/macn003.txt", "medalhão"),
    ("Como José Dias descreve os olhos de Capitu?", "romance/marm08.txt", "oblíqua"),
]

STOP = set("""a o e é de do da dos das em no na nos nas um uma uns umas que quem qual quais
como onde por para com se ao aos à às seu sua seus suas ele ela eles elas lhe não mais
foi era ser são o os as mas ou já muito também quando""".split())


def passagens():
    """Corta cada obra em passagens de ~TAM_PASSAGEM caracteres."""
    saida = []
    with open(RAIZ / "corpus" / "obras.jsonl", encoding="utf-8") as f:
        for linha in f:
            o = json.loads(linha)
            atual = []
            for p in o["paragrafos"]:
                atual.append(p)
                if sum(len(x) for x in atual) >= TAM_PASSAGEM:
                    saida.append({"arquivo": o["arquivo"], "titulo": o["titulo"], "ano": o["ano"],
                                  "texto": "\n".join(atual)})
                    atual = []
            if atual:
                saida.append({"arquivo": o["arquivo"], "titulo": o["titulo"], "ano": o["ano"],
                              "texto": "\n".join(atual)})
    return saida


class Recuperador:
    def __init__(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.passagens = passagens()
        self.vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_df=0.5,
                                   sublinear_tf=True, stop_words=list(STOP),
                                   token_pattern=r"(?u)\b\w\w+\b")
        self.matriz = self.vec.fit_transform(p["texto"] for p in self.passagens)

    def busca(self, pergunta, k=5):
        q = self.vec.transform([pergunta])
        sims = (self.matriz @ q.T).toarray().ravel()
        top = np.argsort(-sims)[:k]
        return [{**self.passagens[i], "score": float(sims[i])} for i in top]


def avalia_recuperacao(rec, ks=(1, 5, 10)):
    res = {f"acerto@{k}": 0 for k in ks} | {f"obra@{k}": 0 for k in ks}
    detalhes = []
    for pergunta, obra, chave in PERGUNTAS:
        top = rec.busca(pergunta, max(ks))
        pos_chave = next((i for i, p in enumerate(top) if chave.lower() in p["texto"].lower()), None)
        pos_obra = next((i for i, p in enumerate(top) if p["arquivo"] == obra), None)
        for k in ks:
            res[f"acerto@{k}"] += pos_chave is not None and pos_chave < k
            res[f"obra@{k}"] += pos_obra is not None and pos_obra < k
        detalhes.append({"pergunta": pergunta, "obra": obra, "chave": chave,
                         "rank_chave": pos_chave, "rank_obra": pos_obra,
                         "top1": {"titulo": top[0]["titulo"], "texto": top[0]["texto"][:300]}})
    n = len(PERGUNTAS)
    return {k: v / n for k, v in res.items()} | {"n_perguntas": n, "detalhes": detalhes}


def monta_prompt(pergunta, top, max_chars=1400):
    contexto, total = [], 0
    for p in top:
        trecho = p["texto"][:max_chars - total]
        contexto.append(f"[{p['titulo']}, {p['ano']}]\n{trecho}")
        total += len(trecho)
        if total >= max_chars:
            break
    return "\n\n".join(contexto) + f"\n\nPergunta: {pergunta}\nResposta:"


def chave_no_prompt(rec, k=3):
    """Em quais perguntas a palavra da resposta chegou ao prompt do gerador."""
    return {p: c.lower() in monta_prompt(p, rec.busca(p, k)).lower() for p, _, c in PERGUNTAS}


def responde(modelo, tok, pergunta, rec, device="cpu", k=3, max_tokens=60):
    import torch
    top = rec.busca(pergunta, k)
    prompt = monta_prompt(pergunta, top)
    ids = tok.encode(prompt)[-(modelo.block_size - max_tokens):]
    x = torch.tensor([ids], dtype=torch.long, device=device)
    torch.manual_seed(1337)
    y = modelo.gera(x, max_tokens, temperature=0.3, top_k=20)
    resposta = tok.decode(y[0, len(ids):].tolist())
    return {"pergunta": pergunta, "fontes": [f"{p['titulo']} ({p['ano']})" for p in top],
            "resposta": next((l.strip() for l in resposta.split("\n") if l.strip()), "")}


def main():
    ap = argparse.ArgumentParser(description="Recuperação + geração sobre a obra de Machado.")
    ap.add_argument("--avaliar", action="store_true", help="mede a recuperação nas 20 perguntas")
    ap.add_argument("--pergunta")
    ap.add_argument("--run", help="modelo gerador (ex.: runs/ptbr_ft); omita para só recuperar")
    ap.add_argument("--gerar-todas", action="store_true", help="gera respostas para as 20 perguntas")
    ap.add_argument("--device", default="cpu")
    args = ap.parse_args()

    rec = Recuperador()
    saida = RAIZ / "results" / "rag.json"
    resultado = json.loads(saida.read_text()) if saida.exists() else {}
    if args.avaliar:
        r = avalia_recuperacao(rec)
        resultado["recuperacao"] = r
        print(json.dumps({k: v for k, v in r.items() if k != "detalhes"}, indent=2))

    if args.pergunta or args.gerar_todas:
        modelo = tok = None
        if args.run:
            from machado_gpt import tokenizacao
            from machado_gpt.avaliacao import NANOGPT, carrega_run
            modelo = carrega_run(args.run, args.device)
            tok = tokenizacao.carrega(NANOGPT / "data" / modelo.dataset)
        perguntas = [args.pergunta] if args.pergunta else [p for p, _, _ in PERGUNTAS]
        respostas = []
        for p in perguntas:
            if modelo is None:
                top = rec.busca(p, 3)
                r = {"pergunta": p, "fontes": [f"{t['titulo']} ({t['ano']})" for t in top],
                     "passagem": top[0]["texto"][:400]}
            else:
                r = responde(modelo, tok, p, rec, args.device)
            respostas.append(r)
            print(json.dumps(r, ensure_ascii=False, indent=2))
        if args.gerar_todas and modelo is not None:
            chaves = {p: c for p, _, c in PERGUNTAS}
            no_prompt = chave_no_prompt(rec)
            for r in respostas:
                r["contem_chave"] = chaves[r["pergunta"]].lower() in r["resposta"].lower()
                r["chave_no_prompt"] = no_prompt[r["pergunta"]]
            acertos = sum(r["contem_chave"] for r in respostas)
            print(f"respostas com a palavra-chave: {acertos}/{len(respostas)}; "
                  f"palavra-chave presente no prompt: {sum(no_prompt.values())}/{len(respostas)}")
            resultado["geracao"] = {"modelo": args.run, "acertos": acertos,
                                    "chave_no_prompt": sum(no_prompt.values()), "respostas": respostas}

    saida.parent.mkdir(exist_ok=True)
    saida.write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")


if __name__ == "__main__":
    main()

In [ ]:
# [AUTORIA: ALUNO]
!python -m machado_gpt.rag --avaliar
if os.path.exists("runs/ptbr_ft"):
    get_ipython().system("python -m machado_gpt.rag --gerar-todas --run runs/ptbr_ft --device cuda | head -60")